In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.cm as cm

print("torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

In [ ]:
DATA_DIR = Path("")
"""My time resolution is not consistent over files, hence i need to adjust it"""
# Group A — moment 0=0s, 2=5s, 3=10s, 4=30s, 5=60s
# Group A — ∆H_10s, ∆H_300s
GROUP_A = ["∆H_10s.xlsx", "∆H_300s.xlsx"]
MOMENT_MAP_A = {0: 0, 2: 5, 4: 30, 6: 60, 8: 90}

# Group B — AMC, MY, ShortCompressions
GROUP_B = ["AMC.xlsx", "MY.xlsx", "ShortCompressions.xlsx"]
MOMENT_MAP_B = {0: 0, 2: 5, 3: 10, 4: 30, 5: 60, 6: 120, 7: 180, 8: 240, 9: 300}

# Group C — all others
GROUP_C = ["F1F2.xlsx", "R1R2R3.xlsx", "FullRecoveryAMC.xlsx", "Forces.xlsx"]
MOMENT_MAP_C = {0: 0, 1: 5, 2: 10, 3: 30, 4: 60, 5: 120, 6: 180}
# Target timepoints for trajectory
TARGET_TIMES = [0, 5, 30, 60]  # seconds

# Features to extract per timepoint
FEATURE_COLS = ["E", "∆H"]

# Experimental parameters (constant per cell)
PARAM_COLS = ["Force of compression", "Contact time"]

In [ ]:
dfs = []

for group, moment_map in [(GROUP_A, MOMENT_MAP_A), 
                           (GROUP_B, MOMENT_MAP_B), 
                           (GROUP_C, MOMENT_MAP_C)]:
    for fname in group:
        fpath = DATA_DIR / fname
        if not fpath.exists():
            print(f"WARNING: {fname} not found, skipping")
            continue
        
        print(f"Loading: {fname}")
        df = pd.read_excel(fpath)
        df["source_file"] = fname
        df = df[df["Moment"].isin(moment_map.keys())].copy()
        df["Time_s"] = df["Moment"].map(moment_map)
        dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows: {len(df_all)}")
print(f"Time points: {sorted(df_all['Time_s'].unique())}")

In [ ]:
# Create unique cell_id that increments every time Moment == 0
df_all["cell_id"] = (df_all["Moment"] == 0).cumsum()

print(f"Unique cells: {df_all['cell_id'].nunique()}")
print(df_all[["cell_id", "Moment", "Time_s", "source_file"]].head(20))

In [ ]:


print(f"Unique cells: {df_all['cell_id'].nunique()}")

# Pivot E and ∆H to wide format
pivot_E = df_all.pivot_table(
    index="cell_id", 
    columns="Time_s", 
    values="E", 
    aggfunc="first"
).add_prefix("E_")

pivot_dH = df_all.pivot_table(
    index="cell_id",
    columns="Time_s",
    values="∆H",
    aggfunc="first"
).add_prefix("dH_")

# Get experimental parameters (one per cell, take from moment 0)
params = df_all[df_all["Time_s"] == 0][
    ["cell_id", "Force of compression", "Contact time", 
     "source_file", "Replicate", "Treatment"]
].set_index("cell_id")

# Combine everything
df_wide = pd.concat([pivot_E, pivot_dH, params], axis=1)
print(f"Wide format shape: {df_wide.shape}")
print(f"Columns: {df_wide.columns.tolist()}")

In [ ]:
# Normalize trajectories by E_0 (baseline stiffness)
df_norm = df_wide_clean.copy()

# Normalize E timepoints by E_0
for col in ["E_5", "E_30", "E_60"]:
    df_norm[col] = df_norm[col] / df_norm["E_0"]

# Normalize dH timepoints by E_0
for col in ["dH_0", "dH_5", "dH_30", "dH_60"]:
    df_norm[col] = df_norm[col] / df_norm["E_0"]

# Drop E_0 — it's 1.0 for every cell now, no information
df_norm = df_norm.drop(columns=["E_0"])

# Verify
print("Feature columns after normalization:")
print(df_norm[["E_5", "E_30", "E_60", "dH_0", "dH_5"]].describe().round(3))

In [ ]:
# Find duplicate cell_ids at moment 0
moment0 = df_all[df_all["Time_s"] == 0]
dupes = moment0[moment0.duplicated(subset="cell_id", keep=False)]
print(f"Duplicate cell_ids at moment 0: {dupes['cell_id'].nunique()}")
print(dupes[["cell_id", "source_file", "Replicate", "Cell number", "Force of compression"]].head(10))

In [ ]:
# Keep only target timepoints
KEEP_COLS = ["E_0", "E_5", "E_30", "E_60",
             "dH_0", "dH_5", "dH_30", "dH_60",
             "Force of compression", "Contact time",
             "source_file", "Replicate", "Treatment"]

df_wide_clean = df_wide[KEEP_COLS].copy()

# Drop cells missing any trajectory timepoint
df_wide_clean = df_wide_clean.dropna(subset=["E_0", "E_5", "E_30", "E_60",
                                              "dH_0", "dH_5", "dH_30", "dH_60"])

print(f"Cells before cleaning: {len(df_wide)}")
print(f"Cells after cleaning: {len(df_wide_clean)}")
print(f"\nCells per file:")
print(df_wide_clean.groupby("source_file").size())
print(f"\nCells per treatment:")
print(df_wide_clean.groupby("Treatment").size())

In [ ]:
df_wide_clean["Treatment"] = df_wide_clean["Treatment"].replace({"Ctrl": "DMSO"})
print(df_wide_clean["Treatment"].value_counts())

In [ ]:
# Separate features from metadata
FEATURE_COLS = ["E_5", "E_30", "E_60", 
                "dH_0", "dH_5", "dH_30", "dH_60"]

METADATA_COLS = ["source_file", "Replicate", "Treatment"]

X = df_norm[FEATURE_COLS].values
meta = df_norm[METADATA_COLS].reset_index(drop=True)

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {FEATURE_COLS}")

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to tensor
X_tensor = torch.FloatTensor(X_scaled)
print(f"\nTensor shape: {X_tensor.shape}")
print(f"Min: {X_tensor.min():.2f}, Max: {X_tensor.max():.2f}, Mean: {X_tensor.mean():.2f}")

In [ ]:
# Create unique batch label
meta["batch"] = meta["source_file"].str.replace(".xlsx", "") + "_R" + meta["Replicate"].astype(str)
print(f"Unique batches: {meta['batch'].nunique()}")
print(meta["batch"].value_counts().sort_index())

# One-hot encode batch
batches = meta["batch"].values
unique_batches = sorted(set(batches))
batch_to_idx = {b: i for i, b in enumerate(unique_batches)}

n_batches = len(unique_batches)
C = np.zeros((len(meta), n_batches))
for i, b in enumerate(batches):
    C[i, batch_to_idx[b]] = 1.0

# Add force and contact time to condition
force = df_norm["Force of compression"].values.reshape(-1, 1)
contact = df_norm["Contact time"].values.reshape(-1, 1)
force_scaled = StandardScaler().fit_transform(force)
contact_scaled = StandardScaler().fit_transform(contact)
C = np.concatenate([C, force_scaled, contact_scaled], axis=1)

C_tensor = torch.FloatTensor(C)
print(f"\nCondition matrix shape: {C_tensor.shape}")  # should be 553 x 22

In [ ]:
"""ARCHITECTURE
input_dim = 10 (features)
condition_dim = 20 (batches)
encoder input = 30 (10 + 20)
latent_dim = 2
decoder input = 22 (2 + 20)
"""


In [ ]:
## Conditional MechanoENV
class CondMechanoVAE(nn.Module):
    def __init__(self, input_dim=7, condition_dim=7, hidden_dim=4, latent_dim=2):
        super(CondMechanoVAE, self).__init__()
        
        # Encoder takes features + condition
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + condition_dim, hidden_dim),
            nn.ReLU()
        )
        
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_var = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder takes z + condition
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + condition_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def encode(self, x, c):
        xc = torch.cat([x, c], dim=1)
        h = self.encoder(xc)
        mu = self.fc_mu(h)
        log_var = self.fc_var(h)
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z, c):
        zc = torch.cat([z, c], dim=1)
        return self.decoder(zc)
    
    def forward(self, x, c):
        mu, log_var = self.encode(x, c)
        z = self.reparameterize(mu, log_var)
        x_reconstructed = self.decode(z, c)
        return x_reconstructed, mu, log_var
    
        """_summary_: we sum instead of multiplying the latent space with 
        the condition, this is a common approach in conditional VAEs and 
        allows the model to learn how to combine the latent representation 
        with the condition in a more flexible way.
        once we concatenated we got a vector of length 14 where at positions 0-6 mechanical features sit
        and at position 7-13 sits the file identity. The linear layer then does: output = [x,c x W +b]
        where W shape 14x4. This means each of the 4 hidden neurons is a weighted sum of all 14 inputs. the network learns by combiation.
        Also the conditions have many zeros and only one 1 hence, by multiplying we would have only one condition contributiog to the latent 
        space, while by summing we allow all conditions to contribute in a more balanced way.
        """

In [ ]:
# Instantiate conditional VAE
device = torch.device("mps")

model = CondMechanoVAE(
    input_dim=7,
    condition_dim=22,
    hidden_dim=32,
    latent_dim=2
).to(device)

print(model)
print(f"\nModel is on device: {next(model.parameters()).device}")

In [ ]:

def vae_loss(x_reconstructed, x_original, mu, log_var):
    reconstruction_loss = nn.functional.mse_loss(
        x_reconstructed, x_original, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return reconstruction_loss + kl_loss


In [ ]:
## Total loss = Reconstruction loss + β × KL loss
def vae_loss(x_reconstructed, x_original, mu, log_var, beta=2):
    reconstruction_loss = nn.functional.mse_loss(x_reconstructed, x_original, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return reconstruction_loss + beta * kl_loss
"""Reconstruction loss wants to memorize every cell's trajectory precisely — it pushes the encoder to use the full latent space freely
KL loss wants to compress everything into a smooth ball around N(0,1) — it pushes all cells toward the same region
"""

In [ ]:
dataset = TensorDataset(X_tensor, C_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 500
losses = []

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    for batch in dataloader:
        x = batch[0].to(device)
        c = batch[1].to(device)
        x_reconstructed, mu, log_var = model(x, c)
        loss = vae_loss(x_reconstructed, x, mu, log_var)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    if epoch % 100 == 0:
        print(f"Epoch {epoch}/{n_epochs} — Loss: {avg_loss:.1f}")

In [ ]:
## train with beta to avoid KL loss dominating early on and collapsing latent space
for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    for batch in dataloader:
        x = batch[0].to(device)
        c = batch[1].to(device)
        x_reconstructed, mu, log_var = model(x, c)
        loss = vae_loss(x_reconstructed, x, mu, log_var, beta=0.1)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    if epoch % 100 == 0:
        print(f"Epoch {epoch}/{n_epochs} — Loss: {avg_loss:.1f}")
        

In [ ]:
### change beta during training

model = CondMechanoVAE(
    input_dim=7,
    condition_dim=22,
    hidden_dim=32,
    latent_dim=2
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
n_epochs = 500

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    
    # Anneal beta from 0 to 1 over first 200 epochs
    beta = min(1.0, epoch / 200)
    
    for batch in dataloader:
        x = batch[0].to(device)
        c = batch[1].to(device)
        x_reconstructed, mu, log_var = model(x, c)
        loss = vae_loss(x_reconstructed, x, mu, log_var, beta=beta)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    if epoch % 100 == 0:
        print(f"Epoch {epoch}/{n_epochs} — β={beta:.2f} — Loss: {avg_loss:.1f}")

In [ ]:
model = CondMechanoVAE(
    input_dim=7,
    condition_dim=22,
    hidden_dim=32,
    latent_dim=2
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []

In [ ]:
# Extract latent space — no UMAP needed, already 2D

### normal VAE
#model.eval()
#with torch.no_grad():
#    X_gpu = X_tensor.to(device)
#    mu, log_var = model.encode(X_gpu)
#    Z = mu.cpu().numpy()
 ## conditional VAE
model.eval()
with torch.no_grad():
    X_gpu = X_tensor.to(device)
    C_gpu = C_tensor.to(device)
    mu, log_var = model.encode(X_gpu, C_gpu)
    Z = mu.cpu().numpy()

print(f"Latent space shape: {Z.shape}")    

print(f"Latent space shape: {Z.shape}")

# Plot colored by treatment
fig, ax = plt.subplots(figsize=(8, 6))
treatments = meta["Treatment"].values
unique_treatments = sorted(set(treatments))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_treatments)))

for treatment, color in zip(unique_treatments, colors):
    mask = treatments == treatment
    ax.scatter(Z[mask, 0], Z[mask, 1],
               label=treatment, color=color, alpha=0.6, s=15, edgecolors="none")

ax.set_xlabel("z1")
ax.set_ylabel("z2")
ax.set_title("MechanoVAE latent space — colored by treatment")
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Extract latent space
model.eval()
with torch.no_grad():
    X_gpu = X_tensor.to(device)
    C_gpu = C_tensor.to(device)
    mu, log_var = model.encode(X_gpu, C_gpu)
    Z = mu.cpu().numpy()

print(f"Latent space shape: {Z.shape}")

# Remove outliers — filter cells > 3 std from mean
Z_mean = Z.mean(axis=0)
Z_std = Z.std(axis=0)
mask_outlier = np.all(np.abs(Z - Z_mean) < 3 * Z_std, axis=1)
Z_clean = Z[mask_outlier]
meta_clean = meta[mask_outlier].reset_index(drop=True)
print(f"Cells after outlier removal: {len(Z_clean)}")

# Plot colored by treatment
fig, ax = plt.subplots(figsize=(8, 6))
treatments = meta_clean["Treatment"].values
unique_treatments = sorted(set(treatments))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_treatments)))

for treatment, color in zip(unique_treatments, colors):
    mask = treatments == treatment
    ax.scatter(Z_clean[mask, 0], Z_clean[mask, 1],
               label=treatment, color=color, alpha=0.6, s=30, edgecolors="none")

ax.set_xlabel("z1")
ax.set_ylabel("z2")
ax.set_title("MechanoVAE latent space — colored by treatment")
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
files = meta_clean["source_file"].values
unique_files = sorted(set(files))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_files)))

for f, color in zip(unique_files, colors):
    mask = files == f
    ax.scatter(Z_clean[mask, 0], Z_clean[mask, 1],
               label=f, color=color, alpha=0.6, s=15, edgecolors="none")

ax.set_xlabel("z1")
ax.set_ylabel("z2")
ax.set_title("MechanoVAE latent space — colored by source file")
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()